# 📈 EcoSense — Exploratory Data Analysis (EDA)
## Comprehensive Analysis of Environmental Sensor Data

This notebook performs a thorough exploration of the generated sensor data, including:

1. **Dataset Overview & Statistics**
2. **Time Series Visualization**
3. **Distribution Analysis**
4. **Correlation Analysis**
5. **Seasonal Decomposition**
6. **Data Quality Assessment**
7. **Shift-based Analysis**

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy import stats as sp_stats

from src.data_generator import SENSOR_CONFIG

# Styling
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11
sns.set_palette('husl')

print('Libraries loaded successfully!')

In [ ]:
# Load data
water_df = pd.read_csv('../data/raw/water_quality.csv', parse_dates=['timestamp'])
air_df = pd.read_csv('../data/raw/air_quality.csv', parse_dates=['timestamp'])
anomaly_log = pd.read_csv('../data/anomalies/injected_anomalies.csv', parse_dates=['start_time', 'end_time'])

print(f'Water Quality: {water_df.shape}')
print(f'Air Quality: {air_df.shape}')
print(f'Anomaly events: {len(anomaly_log)}')

---
## 1. Dataset Overview

In [ ]:
# Water quality sensor columns
water_sensors = ['pH', 'turbidity_ntu', 'dissolved_oxygen_mg_l', 'temperature_c', 'conductivity_us_cm']
air_sensors = ['co2_ppm', 'nox_ppm', 'so2_ppm', 'pm25_ug_m3']

print('=== Water Quality Statistics ===')
water_df[water_sensors].describe().round(3)

In [ ]:
print('=== Air Quality Statistics ===')
air_df[air_sensors].describe().round(4)

In [ ]:
# Data types and info
print('=== Water Quality Data Types ===')
print(water_df.dtypes)
print(f'\nDate range: {water_df["timestamp"].min()} to {water_df["timestamp"].max()}')
print(f'Duration: {(water_df["timestamp"].max() - water_df["timestamp"].min()).days} days')
print(f'Sampling interval: ~{(water_df["timestamp"].diff().median().total_seconds() / 60):.0f} minutes')

---
## 2. Time Series Visualization
### 2.1 Water Quality — Full Time Series

In [ ]:
fig = make_subplots(
    rows=5, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=[f'{s} ({SENSOR_CONFIG["water_quality"][s]["unit"]})' for s in water_sensors]
)

colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B']

for idx, sensor in enumerate(water_sensors):
    config = SENSOR_CONFIG['water_quality'][sensor]
    
    # Main signal
    fig.add_trace(
        go.Scattergl(
            x=water_df['timestamp'], y=water_df[sensor],
            mode='lines', name=sensor,
            line=dict(width=0.5, color=colors[idx]),
            opacity=0.8
        ), row=idx+1, col=1
    )
    
    # Normal range bands
    fig.add_hline(y=config['normal_min'], line_dash='dash', line_color='green',
                  opacity=0.4, row=idx+1, col=1)
    fig.add_hline(y=config['normal_max'], line_dash='dash', line_color='green',
                  opacity=0.4, row=idx+1, col=1)
    
    # Highlight anomalies
    anom_col = f'is_anomaly_{sensor}'
    if anom_col in water_df.columns:
        anom_mask = water_df[anom_col] == 1
        fig.add_trace(
            go.Scattergl(
                x=water_df.loc[anom_mask, 'timestamp'],
                y=water_df.loc[anom_mask, sensor],
                mode='markers', name=f'{sensor} anomaly',
                marker=dict(size=3, color='red', opacity=0.5)
            ), row=idx+1, col=1
        )

fig.update_layout(
    height=1000, title_text='Water Quality Sensors — Complete Time Series',
    showlegend=False,
    template='plotly_white'
)
fig.show()

### 2.2 Air Quality — Full Time Series

In [ ]:
fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    subplot_titles=[f'{s} ({SENSOR_CONFIG["air_quality"][s]["unit"]})' for s in air_sensors]
)

colors_air = ['#E63946', '#457B9D', '#1D3557', '#F4A261']

for idx, sensor in enumerate(air_sensors):
    config = SENSOR_CONFIG['air_quality'][sensor]
    
    fig.add_trace(
        go.Scattergl(
            x=air_df['timestamp'], y=air_df[sensor],
            mode='lines', name=sensor,
            line=dict(width=0.5, color=colors_air[idx]),
            opacity=0.8
        ), row=idx+1, col=1
    )
    
    fig.add_hline(y=config['normal_max'], line_dash='dash', line_color='red',
                  opacity=0.4, row=idx+1, col=1)
    
    anom_col = f'is_anomaly_{sensor}'
    if anom_col in air_df.columns:
        anom_mask = air_df[anom_col] == 1
        fig.add_trace(
            go.Scattergl(
                x=air_df.loc[anom_mask, 'timestamp'],
                y=air_df.loc[anom_mask, sensor],
                mode='markers', name=f'{sensor} anomaly',
                marker=dict(size=3, color='red', opacity=0.5)
            ), row=idx+1, col=1
        )

fig.update_layout(
    height=800, title_text='Air Quality Sensors — Complete Time Series',
    showlegend=False,
    template='plotly_white'
)
fig.show()

---
## 3. Distribution Analysis

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Sensor Value Distributions', fontsize=16, fontweight='bold')

# Water sensors - row 1
for idx, sensor in enumerate(water_sensors):
    ax = axes[0, idx]
    config = SENSOR_CONFIG['water_quality'][sensor]
    
    data = water_df[sensor].dropna()
    ax.hist(data, bins=80, density=True, alpha=0.7, color='steelblue', edgecolor='none')
    
    # Normal range
    ax.axvline(config['normal_min'], color='green', linestyle='--', alpha=0.6, label='Normal')
    ax.axvline(config['normal_max'], color='green', linestyle='--', alpha=0.6)
    
    ax.set_title(f'{sensor}', fontsize=10, fontweight='bold')
    ax.set_xlabel(config['unit'] if config['unit'] else 'Value', fontsize=8)
    
    # Skewness and kurtosis
    skew = data.skew()
    kurt = data.kurtosis()
    ax.text(0.95, 0.95, f'skew={skew:.2f}\nkurt={kurt:.2f}',
            transform=ax.transAxes, fontsize=7, va='top', ha='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Air sensors - row 2
for idx, sensor in enumerate(air_sensors):
    ax = axes[1, idx]
    config = SENSOR_CONFIG['air_quality'][sensor]
    
    data = air_df[sensor].dropna()
    ax.hist(data, bins=80, density=True, alpha=0.7, color='darkorange', edgecolor='none')
    
    ax.axvline(config['normal_max'], color='red', linestyle='--', alpha=0.6, label='Max')
    
    ax.set_title(f'{sensor}', fontsize=10, fontweight='bold')
    ax.set_xlabel(config['unit'], fontsize=8)
    
    skew = data.skew()
    kurt = data.kurtosis()
    ax.text(0.95, 0.95, f'skew={skew:.2f}\nkurt={kurt:.2f}',
            transform=ax.transAxes, fontsize=7, va='top', ha='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Hide unused subplot
axes[1, 4].set_visible(False)

plt.tight_layout()
plt.savefig('../reports/distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Box plots for outlier visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Water quality box plot (normalized for comparison)
water_normalized = water_df[water_sensors].apply(lambda x: (x - x.mean()) / x.std())
axes[0].boxplot([water_normalized[col].dropna() for col in water_sensors],
                labels=[s.replace('_', '\n') for s in water_sensors],
                patch_artist=True,
                boxprops=dict(facecolor='lightblue', alpha=0.7))
axes[0].set_title('Water Quality Sensors (Z-normalized)', fontweight='bold')
axes[0].set_ylabel('Z-Score')
axes[0].axhline(y=3, color='red', linestyle='--', alpha=0.5, label='±3σ')
axes[0].axhline(y=-3, color='red', linestyle='--', alpha=0.5)
axes[0].legend()

# Air quality box plot
air_normalized = air_df[air_sensors].apply(lambda x: (x - x.mean()) / x.std())
axes[1].boxplot([air_normalized[col].dropna() for col in air_sensors],
                labels=[s.replace('_', '\n') for s in air_sensors],
                patch_artist=True,
                boxprops=dict(facecolor='lightyellow', alpha=0.7))
axes[1].set_title('Air Quality Sensors (Z-normalized)', fontweight='bold')
axes[1].set_ylabel('Z-Score')
axes[1].axhline(y=3, color='red', linestyle='--', alpha=0.5, label='±3σ')
axes[1].axhline(y=-3, color='red', linestyle='--', alpha=0.5)
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Correlation Analysis

In [ ]:
# Water quality correlation heatmap
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Water
water_corr = water_df[water_sensors].corr()
mask = np.triu(np.ones_like(water_corr, dtype=bool))
sns.heatmap(water_corr, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, square=True, linewidths=1, ax=axes[0],
            vmin=-1, vmax=1,
            xticklabels=[s.replace('_', ' ') for s in water_sensors],
            yticklabels=[s.replace('_', ' ') for s in water_sensors])
axes[0].set_title('Water Quality — Correlation Matrix', fontweight='bold', fontsize=12)

# Air
air_corr = air_df[air_sensors].corr()
mask_air = np.triu(np.ones_like(air_corr, dtype=bool))
sns.heatmap(air_corr, mask=mask_air, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, square=True, linewidths=1, ax=axes[1],
            vmin=-1, vmax=1,
            xticklabels=[s.replace('_', ' ') for s in air_sensors],
            yticklabels=[s.replace('_', ' ') for s in air_sensors])
axes[1].set_title('Air Quality — Correlation Matrix', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('../reports/correlations.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nKey Correlations (Water):')
for i in range(len(water_sensors)):
    for j in range(i+1, len(water_sensors)):
        corr_val = water_corr.iloc[i, j]
        if abs(corr_val) > 0.3:
            print(f'  {water_sensors[i]} ↔ {water_sensors[j]}: {corr_val:.3f}')

---
## 5. Seasonal Decomposition

Decompose the time series into **trend**, **seasonal**, and **residual** components.

In [ ]:
# Seasonal decomposition for key sensors
decompose_sensors = {
    'Water Temperature': ('water', 'temperature_c'),
    'Water pH': ('water', 'pH'),
    'Air CO₂': ('air', 'co2_ppm'),
    'Air PM2.5': ('air', 'pm25_ug_m3')
}

for title, (data_type, sensor) in decompose_sensors.items():
    df = water_df if data_type == 'water' else air_df
    
    # Resample to hourly for cleaner decomposition
    ts = df.set_index('timestamp')[sensor].resample('1h').mean().dropna()
    
    # Decompose with daily period (24 hours)
    decomposition = seasonal_decompose(ts, model='additive', period=24)
    
    fig, axes = plt.subplots(4, 1, figsize=(16, 10), sharex=True)
    fig.suptitle(f'Seasonal Decomposition — {title}', fontsize=14, fontweight='bold')
    
    decomposition.observed.plot(ax=axes[0], color='steelblue', linewidth=0.5)
    axes[0].set_ylabel('Observed')
    
    decomposition.trend.plot(ax=axes[1], color='darkorange', linewidth=1)
    axes[1].set_ylabel('Trend')
    
    decomposition.seasonal.plot(ax=axes[2], color='green', linewidth=0.5)
    axes[2].set_ylabel('Seasonal')
    
    decomposition.resid.plot(ax=axes[3], color='red', linewidth=0.5)
    axes[3].set_ylabel('Residual')
    
    plt.tight_layout()
    plt.savefig(f'../reports/decomposition_{sensor}.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 6. Data Quality Assessment

In [ ]:
# Missing data analysis
print('=== Missing Data Summary ===')
print('\nWater Quality:')
water_missing = water_df[water_sensors].isnull().sum()
water_missing_pct = (water_missing / len(water_df) * 100).round(3)
missing_summary_w = pd.DataFrame({
    'Missing Count': water_missing,
    'Missing %': water_missing_pct,
    'Completeness %': (100 - water_missing_pct).round(3)
})
print(missing_summary_w)

print('\nAir Quality:')
air_missing = air_df[air_sensors].isnull().sum()
air_missing_pct = (air_missing / len(air_df) * 100).round(3)
missing_summary_a = pd.DataFrame({
    'Missing Count': air_missing,
    'Missing %': air_missing_pct,
    'Completeness %': (100 - air_missing_pct).round(3)
})
print(missing_summary_a)

In [ ]:
# Missing data heatmap (sample 1000 rows for visibility)
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

sample_idx = np.linspace(0, len(water_df)-1, 1000, dtype=int)

sns.heatmap(water_df[water_sensors].iloc[sample_idx].isnull().T, 
            cbar=False, yticklabels=True, ax=axes[0],
            cmap=['#2ecc71', '#e74c3c'])
axes[0].set_title('Water Quality — Missing Data Pattern', fontweight='bold')
axes[0].set_xlabel('Sample Index')

sns.heatmap(air_df[air_sensors].iloc[sample_idx].isnull().T,
            cbar=False, yticklabels=True, ax=axes[1],
            cmap=['#2ecc71', '#e74c3c'])
axes[1].set_title('Air Quality — Missing Data Pattern', fontweight='bold')
axes[1].set_xlabel('Sample Index')

plt.tight_layout()
plt.savefig('../reports/missing_data.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Shift-based Analysis

Compare sensor behavior across factory shifts:
- **Morning Shift** (06:00–14:00)
- **Afternoon Shift** (14:00–22:00)
- **Night Shift** (22:00–06:00)

In [ ]:
# Add shift labels
def get_shift(hour):
    if 6 <= hour < 14:
        return 'Morning (06-14)'
    elif 14 <= hour < 22:
        return 'Afternoon (14-22)'
    else:
        return 'Night (22-06)'

water_df['hour'] = water_df['timestamp'].dt.hour
water_df['shift'] = water_df['hour'].apply(get_shift)

fig, axes = plt.subplots(1, len(water_sensors), figsize=(20, 5))
fig.suptitle('Water Quality by Factory Shift', fontsize=14, fontweight='bold')

shift_colors = {'Morning (06-14)': '#F4A261', 'Afternoon (14-22)': '#2A9D8F', 'Night (22-06)': '#264653'}

for idx, sensor in enumerate(water_sensors):
    data_by_shift = [water_df[water_df['shift'] == s][sensor].dropna() for s in shift_colors.keys()]
    bp = axes[idx].boxplot(data_by_shift, labels=['Morning', 'Afternoon', 'Night'],
                          patch_artist=True, widths=0.6)
    for patch, color in zip(bp['boxes'], shift_colors.values()):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    axes[idx].set_title(sensor.replace('_', ' '), fontsize=9, fontweight='bold')
    axes[idx].tick_params(axis='x', labelsize=7)

plt.tight_layout()
plt.savefig('../reports/shift_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Clean up temp columns
water_df.drop(columns=['hour', 'shift'], inplace=True, errors='ignore')

In [ ]:
# Hourly profile (average by hour of day)
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Water — hourly profile
water_hourly = water_df.copy()
water_hourly['hour'] = water_hourly['timestamp'].dt.hour
for sensor in water_sensors:
    hourly_mean = water_hourly.groupby('hour')[sensor].mean()
    hourly_normalized = (hourly_mean - hourly_mean.mean()) / hourly_mean.std()
    axes[0].plot(hourly_normalized.index, hourly_normalized.values, 
                marker='o', markersize=4, label=sensor.replace('_', ' '), linewidth=1.5)

axes[0].set_title('Water Quality — Hourly Profile (Normalized)', fontweight='bold')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Z-Score')
axes[0].legend(fontsize=8, ncol=3)
axes[0].set_xticks(range(24))
axes[0].axvspan(6, 14, alpha=0.1, color='yellow', label='Morning Shift')
axes[0].axvspan(14, 22, alpha=0.1, color='blue', label='Afternoon Shift')

# Air — hourly profile
air_hourly = air_df.copy()
air_hourly['hour'] = air_hourly['timestamp'].dt.hour
for sensor in air_sensors:
    hourly_mean = air_hourly.groupby('hour')[sensor].mean()
    hourly_normalized = (hourly_mean - hourly_mean.mean()) / hourly_mean.std()
    axes[1].plot(hourly_normalized.index, hourly_normalized.values,
                marker='s', markersize=4, label=sensor.replace('_', ' '), linewidth=1.5)

axes[1].set_title('Air Quality — Hourly Profile (Normalized)', fontweight='bold')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Z-Score')
axes[1].legend(fontsize=8, ncol=2)
axes[1].set_xticks(range(24))
axes[1].axvspan(6, 14, alpha=0.1, color='yellow')
axes[1].axvspan(14, 22, alpha=0.1, color='blue')

plt.tight_layout()
plt.savefig('../reports/hourly_profile.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ EDA complete! Proceed to notebook 03_anomaly_detection.ipynb')